# Explore the ERA5 Dataset with Python

intinya, tetep aja harus request data -> gak readily available. Jadi datanya harus distore somewhere. 

In [ ]:
import cdsapi
import earthkit.data as ekd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import os
import xarray as xr 

from zipfile import ZipFile

In [ ]:
variables = [
        "mean_wave_direction",
        "significant_height_of_combined_wind_waves_and_swell",
        "peak_wave_period"
    ]

areas = np.array([
        [  59.2, -141.7,   58.8, -141.3],
        [  38.2, -124.2,   37.8, -123.8],
        [  40.7,  -71.2,   40.3,  -70.8],
        [  53.2,  -11.2,   52.8,  -10.8],
        [  41.7,  142.8,   41.3,  143.2],
        [  13.2,  -92.7,   12.8,  -92.3],
        [  25.2,  -96.7,   24.8,  -96.3],
        [  36.7,   13.8,   36.3,   14.2],
        [   6.7,  -52.7,    6.3,  -52.3],
        [   5.7,  -12.2,    5.3,  -11.8],
        [  19.7,   58.8,   19.3,   59.2],
        [  21.2,   89.3,   20.8,   89.7],
        [  28.2,  127.3,   27.8,  127.7],
        [ -21.3,  -71.7,  -21.7,  -71.3],
        [ -23.8,  -41.7,  -24.2,  -41.3],
        [ -10.8,   12.8,  -11.2,   13.2],
        [  -2.8,   40.8,   -3.2,   41.2],
        [  -4.8,  100.8,   -5.2,  101.2],
        [  -1.8,  141.3,   -2.2,  141.7],
        [ -53.8,  -75.2,  -54.2,  -74.8],
        [ -35.3,   17.8,  -35.7,   18.2],
        [ -34.8,  113.8,  -35.2,  114.2],
        [ -31.3,  153.3,  -31.7,  153.7],
        [ -44.3,  172.3,  -44.7,  172.7]
    ])

priorities = [2, 6, 10, 17, 19, 22]
priority_areas = areas[priorities,:]

In [ ]:
import matplotlib.pyplot as plt

# Function to plot a rectangle at (ox, oy)
def plot_rect(ox, oy, ax, color_sel='None'):
    rect = plt.Rectangle((ox, oy), 5, 5, linewidth=1, edgecolor=color_sel, facecolor=color_sel)  # Customize color if needed
    ax.add_patch(rect)


fig, axs = plt.subplots(1, 1)

# Loop through priority areas and plot rectangles
for o in areas:
    plot_rect(o[1], o[0], axs, 'b')

# Loop through priority areas and plot rectangles
for o in priority_areas:
    plot_rect(o[1], o[0], axs, 'r')

# Optional: setting axis limits to fit all rectangles
axs.set_xlim(-180, 180)
axs.set_ylim(-90, 90)

# Adding grid and labels for clarity
axs.grid(True)
axs.set_title('Priority Areas')
axs.set_xlabel('X Coordinate')
axs.set_ylabel('Y Coordinate')

# Show the plot
plt.show()

In [ ]:
variables

In [ ]:
priorities = [2, 6, 10, 17, 19, 22]
nonpriorities = [0, 1, 3, 4, 5, 7, 8, 9, 11, 12, 13, 14, 15, 16, 18, 21, 23]
sandy = [1, 2, 4, 5, 6, 7, 9, 10, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23]
sandy_priorities = [2, 6, 10, 17, 19, 22]
sandy_nonpriorities = [1, 4, 5, 7, 9, 13, 14, 15, 16, 20, 21, 23]

In [ ]:
[sandy_nonpriorities[3]]

In [ ]:
dataset = "reanalysis-era5-single-levels"
client = cdsapi.Client()

for i in [sandy_nonpriorities[3]]:#sandy_nonpriorities[3:]:
    for year in range(1991, 2020, 4):
        request = {
            "product_type": ["reanalysis"],
            "variable": ['significant_height_of_combined_wind_waves_and_swell'],
            "year": [str(year), str(year+1), str(year+2), str(year+3)],
            "month": [
                "01", "02", "03",
                "04", "05", "06",
                "07", "08", "09",
                "10", "11", "12"
            ],
            "day": [
                "01", "02", "03",
                "04", "05", "06",
                "07", "08", "09",
                "10", "11", "12",
                "13", "14", "15",
                "16", "17", "18",
                "19", "20", "21",
                "22", "23", "24",
                "25", "26", "27",
                "28", "29", "30",
                "31"
            ],
            "time": [
                "00:00", "01:00", "02:00",
                "03:00", "04:00", "05:00",
                "06:00", "07:00", "08:00",
                "09:00", "10:00", "11:00",
                "12:00", "13:00", "14:00",
                "15:00", "16:00", "17:00",
                "18:00", "19:00", "20:00",
                "21:00", "22:00", "23:00",
            ],
            "area": areas[i].tolist(),
            "data_format": "grib",
            "download_format": "unarchived"
        }

        target = f'../data/ERA5/p{i+1}_{year}_{year+3}.grib'
        client.retrieve(dataset, request, target)


In [ ]:
ds = xr.load_dataarray('../data/ERA5/p20_1989.grib', engine='cfgrib')
ds = ds.squeeze()

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(ds.time.values, ds.values)

In [ ]:
import os
from glob import glob

In [ ]:
# Define the directory and the file pattern (e.g., p20_*.grb)
input_dir = '../data/ERA5/'
file_pattern = 'p3_*.grib'

files = glob(os.path.join(input_dir, file_pattern))

datasets = [
    xr.open_dataset(f, engine='cfgrib')  # or with options, see below
    for f in files
]

ds = xr.concat(dataset, dim='time')

In [ ]:
hs = ds.squeeze().swh.values
time = ds.time.values

plt.figure(figsize=(10,6))
plt.plot(time, hs)

## Time series dataset
ERA5 have a dedicated catalogue entry, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way)

However, the retrieved .nc file can't be opened. The dataset is still experimental.  
More info: [ERA5 hourly time-series data on single levels from 1940 to present](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)

In [ ]:
poi = gpd.read_file('../data/point_of_interest.json')

lon = poi.iloc[0].geometry.x
lat = poi.iloc[0].geometry.y

print(lon, lat)
# # small helper function to round the coordinates according to ERA5 grid 
# def round_to_grid(value, grid_size):
#     return round(value / grid_size) * grid_size

# # ERA5 has grid size of 0.25 degrees
# lon_g = round_to_grid(lon, 0.25)
# lat_g = round_to_grid(lat, 0.25)

# print(lon_g, lat_g)

In [ ]:
import cdsapi

dataset = "reanalysis-era5-single-levels-timeseries"
client = cdsapi.Client()

for i in range(len(poi)):
    request = {
        "variable": [
            "mean_wave_direction",
            "mean_wave_period",
            "significant_height_of_combined_wind_waves_and_swell"
        ],
        "location": {"longitude": poi.iloc[i].geometry.x, "latitude": poi.iloc[i].geometry.y},
        "date": ["1979-01-01/2020-12-31"],
        "data_format": "netcdf"
    }

    target = f'../data/ERA5/ts/p{i+1}.zip'
    client.retrieve(dataset, request, target)


In [ ]:
# unzip the files into one directory and rename them according to the location 
# Define the common extraction directory
extraction_path = '../data/ERA5/ts/unzipped/'

# Ensure the extraction directory exists
os.makedirs(extraction_path, exist_ok=True)

# Loop through p1 to p24
for i in range(1, 25):
    # Define the source zip file for each p1 to p24
    zip_file_path = f'../data/ERA5/ts/p{i}.zip'

    # Open the zip file and extract the contents
    with ZipFile(zip_file_path, 'r') as zObject:
        # Extract all files into the common extraction directory
        zObject.extractall(path=extraction_path)

        # List the files in the extraction path to find the extracted file
        extracted_files = os.listdir(extraction_path)

        # Loop over the files and rename the matching one
        for extracted_file in extracted_files:
            # Check if the file matches the pattern
            if extracted_file.startswith('reanalysis-era5-single-levels-timeseries-') and extracted_file.endswith('.nc'):
                # Define the new name for the file
                new_file_name = f'p{i}_ts.nc'
                
                # Construct full paths for the old and new file names
                old_file_path = os.path.join(extraction_path, extracted_file)
                new_file_path = os.path.join(extraction_path, new_file_name)
                
                # Rename the file
                os.rename(old_file_path, new_file_path)
                print(f"File {extracted_file} renamed to: {new_file_name}")
                break  # Since there's only one file, we can break after renaming
